<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%204/Aula_4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4.1 — O que são workflows e quando usar

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 4 — Orquestrando agentes com workflows

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 3** construímos um time conversacional do ScoutAI FC:
- Treinador-líder
- Olheiro
- Analista

Nova demanda:

**Ex.: Recomendação de escalação**:
```
1. Coletar dados dos jogadores
2. Analisar forma
3. Decidir escalação
4. Apresentar
```

Plano da aula:

1. **Conceito de workflow** em poucas linhas
2. Implementar o **workflow mínimo**


## Setup


In [ ]:
!pip install -U agno openai tavily-python wikipedia

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — Reconstruindo os agentes da Aula 3



In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.team import Team
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

# Modelos por agente (boa prática estabelecida na Aula 3.3)
modelo_olheiro = OpenAIChat(id="gpt-5.4")
modelo_treinador = OpenAIChat(id="gpt-5.4")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas e em base interna oficial",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável.",
        "Você tem duas fontes disponíveis:",
        #"• BASE INTERNA (Knowledge): regulamento oficial da Copa do Mundo 2026 — formato, critérios de desempate, regras de fase eliminatória.",
        "• Tavily (web): eventos recentes — últimos jogos, convocações, forma de jogadores.",
        "• Wikipedia: fatos históricos consolidados — Copas antigas, biografias, técnicos do passado.",
        #"Sempre que a pergunta envolver REGRAS OFICIAIS da Copa 2026, busque PRIMEIRO na base interna — é a fonte autoritativa.",
        "Para forma atual, use Tavily. Para histórico, use Wikipedia. Combine fontes quando a pergunta tiver múltiplas necessidades.",
        "Retorne dados objetivos — não interprete nem opine.",
        "Se a busca falhar em todas as fontes, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    #knowledge=knowledge,            # ← peça nova: base interna como knowledge
    #search_knowledge=True,          # ← permite ao agente buscar na base
    markdown=True,
)

# Treinador agora é um Agent comum, não líder de time.
# Ele recebe os dados da etapa anterior e produz a resposta final ao usuário.
treinador = Agent(
    name="Treinador do ScoutAI FC ",
    role="Sintetiza dados e produz recomendação tática para o usuário",
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC — uma plataforma de IA dedicada à Seleção Brasileira masculina de futebol.",
        "Sua especialidade é a Seleção: história, conquistas, jogadores, técnicos, táticas e cultura ao redor do time.",
        "Seu público são torcedores, analistas e comissões técnicas. Adapte o nível técnico ao tipo de pergunta.",

        "Sua função: produzir uma recomendação de escalação clara e estruturada.",
        "Sempre inclua: formação sugerida (ex: 4-3-3), 11 titulares com posição, "
        "principais reservas e justificativa tática em 2-3 frases.",
        "Responda em português do Brasil, com tom profissional.",
    ],
    markdown=True,
)

---

## Passo 2 — O que é um workflow?

```
Pergunta do usuário
        │
        ▼
   ETAPA 1: Olheiro coleta dados
        │ (output → input)
        ▼
   ETAPA 2: Treinador sintetiza e apresenta
        │
        ▼
   Resposta
```

A diferença em relação ao time:

| Aspecto | Time conversacional (Aula 3) | Workflow (esta aula) |
|---|---|---|
| **Quem decide o fluxo** | LLM (líder do time) | Código |
| **Ordem das etapas** | Variável a cada execução | Fixa, sempre a mesma |
| **Previsibilidade** | Baixa-média | Alta |
| **Flexibilidade** | Alta | Baixa-média |
| **Auditabilidade** | Difícil | Direta |
| **Quando usar** | Conversa, perguntas exploratórias | Tarefas estruturadas, decisões repetidas |



---

## Passo 3 — Construindo o workflow mínimo

O Agno tem duas peças centrais pra workflow:

- **`Step`** — uma etapa. Aceita `agent=...` (agente que executa), `team=...` (time que executa) ou `executor=função` (função Python custom)
- **`Workflow`** — o orquestrador. Recebe lista de `Step` em `steps=[...]`, executa em ordem.



In [ ]:
from agno.workflow import Step, Workflow

workflow_escalacao = Workflow(
    name="Recomendação de Escalação",
    steps=[
        Step(name="Coleta de dados", agent=olheiro),
        Step(name="Síntese da recomendação", agent=treinador),
    ],
)

---

## Passo 4 — O workflow em ação



In [ ]:
workflow_escalacao.print_response(
"Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), "
"considerando forma física atual dos convocados e o adversário. Exclua da lista jogadores lesionados.",
    stream=True,
)


### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Time conversacional (Aula 3) — para perguntas exploratórias
├── ✅ Workflow mínimo (esta aula) — para tarefas estruturadas
│   ├── Step 1: Coleta (Olheiro)
│   └── Step 2: Síntese (Treinador)
│
├── ❌ Sem agente especialista em decidir escalação      → Aula 4.2 (Auxiliar Técnico)
├── ❌ Sem contrato de tipos entre etapas (Pydantic)     → Aula 4.2
├── ❌ Sem lógica condicional, paralela ou iterativa     → Aula 4.3
└── ❌ Sem roteamento entre time e workflow              → Aula 4.3
```

### Próxima aula

**Aula 4.2 — Construindo o workflow completo**

